# Experiment 3: Regularization breaks residual symmetry

After activation functions reduce $\mathrm{GL}(n)$ to a smaller centralizer, regularization breaks the symmetry further by penalizing some orbit members over others. Different regularizers select different *preferred representatives* from each equivalence class, and the residual symmetry group depends on the geometry of the regularization norm.

We demonstrate this with a **linear autoencoder** (encoder $f(x) = Wx$, decoder $g(h) = W'h$), where the latent-space symmetry is $\mathrm{GL}_d(\mathbb{R})$. Any invertible $K$ can be inserted: $W \mapsto KW$, $W' \mapsto W'K^{-1}$ without changing reconstruction. We train under three regimes:

| Regularization | Expected residual group | Expected weight structure |
|---|---|---|
| None | $\mathrm{GL}_d(\mathbb{R})$ | Arbitrary (no preferred factorization) |
| $L_2$ (Frobenius) | $O_d(\mathbb{R})$ | $W' \approx W^\top$ (Transpose Theorem) |
| $L_1$ (entry-wise) | $B_d = \mathbb{Z}_2^d \rtimes S_d$ | Sparse, axis-aligned |


In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
})

torch.manual_seed(42)
np.random.seed(42)

# Generate synthetic data: points on a k-dimensional subspace of R^n
n_dim = 8       # ambient dimension
k_dim = 3       # latent dimension (autoencoder bottleneck)
n_samples = 500

# Random subspace
V = torch.linalg.qr(torch.randn(n_dim, k_dim, dtype=torch.float64))[0]
coeffs = torch.randn(n_samples, k_dim, dtype=torch.float64)
noise = 0.05 * torch.randn(n_samples, n_dim, dtype=torch.float64)
X = coeffs @ V.T + noise

print(f"Data: {n_samples} points in R^{n_dim}, lying near a {k_dim}-dim subspace")
print(f"Data shape: {X.shape}")


Data: 500 points in R^8, lying near a 3-dim subspace
Data shape: torch.Size([500, 8])


## 1. Training linear autoencoders under different regularizations

We train three linear autoencoders with identical architecture but different regularization:
- **No regularization**: $\mathcal{L} = \|x - W'Wx\|^2$
- **$L_2$ regularization**: $\mathcal{L} = \|x - W'Wx\|^2 + \lambda(\|W\|_F^2 + \|W'\|_F^2)$
- **$L_1$ regularization**: $\mathcal{L} = \|x - W'Wx\|^2 + \lambda(\|W\|_1 + \|W'\|_1)$


In [2]:
class LinearAutoencoder(nn.Module):
    def __init__(self, n_dim, k_dim):
        super().__init__()
        self.encoder = nn.Linear(n_dim, k_dim, bias=False, dtype=torch.float64)
        self.decoder = nn.Linear(k_dim, n_dim, bias=False, dtype=torch.float64)
    
    def forward(self, x):
        return self.decoder(self.encoder(x))

def train_autoencoder(X, n_dim, k_dim, reg_type='none', lam=0.01, n_epochs=2000, lr=0.005):
    model = LinearAutoencoder(n_dim, k_dim)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    losses = []
    for epoch in range(n_epochs):
        recon = model(X)
        recon_loss = ((X - recon) ** 2).mean()
        
        if reg_type == 'l2':
            reg = lam * (model.encoder.weight.norm()**2 + model.decoder.weight.norm()**2)
        elif reg_type == 'l1':
            reg = lam * (model.encoder.weight.abs().sum() + model.decoder.weight.abs().sum())
        else:
            reg = 0.0
        
        loss = recon_loss + reg
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    
    return model, losses

# Train all three
print("Training with no regularization...")
model_none, losses_none = train_autoencoder(X, n_dim, k_dim, 'none', lam=0.0)
print("Training with L2 regularization...")
model_l2, losses_l2 = train_autoencoder(X, n_dim, k_dim, 'l2', lam=0.01)
print("Training with L1 regularization...")
model_l1, losses_l1 = train_autoencoder(X, n_dim, k_dim, 'l1', lam=0.005)
print("Done.")


Training with no regularization...


Training with L2 regularization...


Training with L1 regularization...


Done.


## 2. Training curves

In [3]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(losses_none, label='No regularization', color='#555555', alpha=0.8)
ax.plot(losses_l2, label='$L_2$ (Frobenius)', color='#2E86C1', alpha=0.8)
ax.plot(losses_l1, label='$L_1$ (entry-wise)', color='#C0392B', alpha=0.8)
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Training curves: linear autoencoder under different regularizations')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('Ex3_fig_reg_training_curves.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex3_fig_reg_training_curves.png")


Saved: Ex3_fig_reg_training_curves.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_30871/1500762489.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. The Transpose Theorem: $L_2$ regularization yields $W' \approx W^\top$

Under $L_2$ regularization, all critical points of the linear autoencoder satisfy $W' = W^\top$ (Kunin et al., 2019). This is not a coincidence — it is the natural consequence of the $O(d)$ residual symmetry that Frobenius regularization imposes. The orthogonal group treats $W$ and $W^\top$ as equivalent, and the regularizer selects the symmetric representative.

We verify this by comparing $W'$ with $W^\top$ for each trained model.


In [4]:
W_none = model_none.encoder.weight.detach().numpy()   # shape: (k, n)
Wp_none = model_none.decoder.weight.detach().numpy()  # shape: (n, k)

W_l2 = model_l2.encoder.weight.detach().numpy()
Wp_l2 = model_l2.decoder.weight.detach().numpy()

W_l1 = model_l1.encoder.weight.detach().numpy()
Wp_l1 = model_l1.decoder.weight.detach().numpy()

# Check: is W' ≈ W^T?  Compare Wp (n x k) with W^T (n x k)
def transpose_error(W, Wp):
    # Normalized error between W' and W^T
    WT = W.T
    return np.linalg.norm(Wp - WT) / np.linalg.norm(WT)

print("Transpose Theorem verification: ||W' - W^T|| / ||W^T||")
print(f"  No regularization: {transpose_error(W_none, Wp_none):.4f}")
print(f"  L2 regularization: {transpose_error(W_l2, Wp_l2):.4f}")
print(f"  L1 regularization: {transpose_error(W_l1, Wp_l1):.4f}")
print()
print("Under L2, W' ≈ W^T (small error). Without regularization, W' and W^T")
print("can differ substantially — the GL(d) freedom allows arbitrary factorizations.")


Transpose Theorem verification: ||W' - W^T|| / ||W^T||
  No regularization: 0.6354
  L2 regularization: 0.0000
  L1 regularization: 0.5911

Under L2, W' ≈ W^T (small error). Without regularization, W' and W^T
can differ substantially — the GL(d) freedom allows arbitrary factorizations.


## 4. Weight structure visualization

We visualize the encoder weight $W$, decoder weight $W'$, and the product $W'W$ (which should approximate the projection onto the data subspace regardless of regularization).


In [5]:
fig, axes = plt.subplots(3, 3, figsize=(14, 11))

titles_row = ['No regularization', '$L_2$ (Frobenius)', '$L_1$ (entry-wise)']
titles_col = ['Encoder $W$', 'Decoder $W\'$', '$W\'W$ (projection)']
weights = [
    (W_none, Wp_none, Wp_none @ W_none),
    (W_l2, Wp_l2, Wp_l2 @ W_l2),
    (W_l1, Wp_l1, Wp_l1 @ W_l1),
]

for col, (W, Wp, WpW) in enumerate(weights):
    for row, (mat, title_c) in enumerate(zip([W, Wp, WpW], titles_col)):
        ax = axes[row, col]
        vmax = max(abs(mat.min()), abs(mat.max()))
        im = ax.imshow(mat, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
        if row == 0:
            ax.set_title(titles_row[col], fontsize=13, fontweight='bold')
        if col == 0:
            ax.set_ylabel(title_c, fontsize=12)
        plt.colorbar(im, ax=ax, shrink=0.8)

plt.suptitle('Weight matrices under different regularizations', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('Ex3_fig_reg_weight_matrices.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex3_fig_reg_weight_matrices.png")


Saved: Ex3_fig_reg_weight_matrices.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_30871/1376313581.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Measuring orthogonality and sparsity

To quantify the structural differences:
- **Orthogonality**: $\|W^\top W - I\|_F / \|I\|_F$ (how close $W$ is to having orthonormal rows)
- **Sparsity**: fraction of entries in $W$ with $|w_{ij}| < 0.01 \cdot \max|w_{ij}|$


In [6]:
def orthogonality_measure(W):
    # How close is W to having orthonormal rows? ||W W^T - I||_F / ||I||_F
    WWT = W @ W.T
    I = np.eye(WWT.shape[0])
    return np.linalg.norm(WWT - I) / np.linalg.norm(I)

def sparsity_measure(W, threshold_frac=0.01):
    # Fraction of entries below threshold_frac * max(|W|)
    threshold = threshold_frac * np.abs(W).max()
    return (np.abs(W) < threshold).mean()

print(f"{'Metric':<25} | {'No reg':>10} | {'L2':>10} | {'L1':>10}")
print("-" * 62)

orth_none = orthogonality_measure(W_none)
orth_l2 = orthogonality_measure(W_l2)
orth_l1 = orthogonality_measure(W_l1)
print(f"{'Orthogonality error':<25} | {orth_none:>10.4f} | {orth_l2:>10.4f} | {orth_l1:>10.4f}")

sp_none = sparsity_measure(W_none)
sp_l2 = sparsity_measure(W_l2)
sp_l1 = sparsity_measure(W_l1)
print(f"{'Sparsity (encoder W)':<25} | {sp_none:>10.2%} | {sp_l2:>10.2%} | {sp_l1:>10.2%}")

sp_none_d = sparsity_measure(Wp_none)
sp_l2_d = sparsity_measure(Wp_l2)
sp_l1_d = sparsity_measure(Wp_l1)
decoder_label = "Sparsity (decoder W')"
print(f"{decoder_label:<25} | {sp_none_d:>10.2%} | {sp_l2_d:>10.2%} | {sp_l1_d:>10.2%}")

te_none = transpose_error(W_none, Wp_none)
te_l2 = transpose_error(W_l2, Wp_l2)
te_l1 = transpose_error(W_l1, Wp_l1)
print(f"{'Transpose error':<25} | {te_none:>10.4f} | {te_l2:>10.4f} | {te_l1:>10.4f}")


Metric                    |     No reg |         L2 |         L1
--------------------------------------------------------------
Orthogonality error       |     0.3668 |     0.0840 |     1.0516
Sparsity (encoder W)      |      0.00% |      0.00% |     83.33%
Sparsity (decoder W')     |      8.33% |      0.00% |     16.67%
Transpose error           |     0.6354 |     0.0000 |     0.5911


## 6. Visualizing the residual symmetry

To make the residual symmetry concrete, we test whether the trained weights are invariant under different transformations applied to the latent space.

For each model, we apply:
- A random **orthogonal** matrix $Q$ (rotation/reflection): $W \mapsto QW$, $W' \mapsto W'Q^{-1}$
- A random **signed permutation** (hyperoctahedral element): same reparametrization
- A random **general invertible** matrix $M$: same reparametrization

We measure how much the regularization penalty changes under each transformation. If the penalty is invariant, the transformation is in the residual symmetry group.


In [7]:
def frobenius_penalty(W, Wp):
    return np.linalg.norm(W, 'fro')**2 + np.linalg.norm(Wp, 'fro')**2

def l1_penalty(W, Wp):
    return np.abs(W).sum() + np.abs(Wp).sum()

def random_orthogonal(d):
    M = np.random.randn(d, d)
    Q, _ = np.linalg.qr(M)
    return Q

def random_signed_permutation(d):
    perm = np.random.permutation(d)
    signs = np.random.choice([-1, 1], size=d)
    M = np.zeros((d, d))
    M[np.arange(d), perm] = signs
    return M

def random_invertible(d):
    M = np.random.randn(d, d)
    while abs(np.linalg.det(M)) < 0.1:
        M = np.random.randn(d, d)
    return M

n_trials = 100
results = {}

for name, W, Wp in [('No reg', W_none, Wp_none), ('L2', W_l2, Wp_l2), ('L1', W_l1, Wp_l1)]:
    for penalty_name, penalty_fn in [('Frobenius', frobenius_penalty), ('L1', l1_penalty)]:
        base_penalty = penalty_fn(W, Wp)
        for transform_name, transform_fn in [('Orthogonal', random_orthogonal), 
                                               ('Signed perm', random_signed_permutation),
                                               ('General inv', random_invertible)]:
            changes = []
            for _ in range(n_trials):
                K = transform_fn(k_dim)
                K_inv = np.linalg.inv(K)
                W_new = K @ W
                Wp_new = Wp @ K_inv
                new_penalty = penalty_fn(W_new, Wp_new)
                changes.append(abs(new_penalty - base_penalty) / (base_penalty + 1e-10))
            key = (name, penalty_name, transform_name)
            results[key] = np.mean(changes)

# Display as table
print("Relative change in regularization penalty under latent-space reparametrization")
print("(0.0 = invariant = transformation is in the residual symmetry group)\n")

for penalty_name in ['Frobenius', 'L1']:
    print(f"=== {penalty_name} penalty ===")
    print(f"{'Transform':<16} | {'No reg':>10} | {'L2 trained':>10} | {'L1 trained':>10}")
    print("-" * 55)
    for transform_name in ['Orthogonal', 'Signed perm', 'General inv']:
        vals = [results[(m, penalty_name, transform_name)] for m in ['No reg', 'L2', 'L1']]
        print(f"{transform_name:<16} | {vals[0]:>10.4f} | {vals[1]:>10.4f} | {vals[2]:>10.4f}")
    print()


Relative change in regularization penalty under latent-space reparametrization
(0.0 = invariant = transformation is in the residual symmetry group)

=== Frobenius penalty ===
Transform        |     No reg | L2 trained | L1 trained
-------------------------------------------------------
Orthogonal       |     0.0000 |     0.0000 |     0.0000
Signed perm      |     0.0000 |     0.0000 |     0.0000
General inv      |     7.4499 |    16.8490 |     6.6056

=== L1 penalty ===
Transform        |     No reg | L2 trained | L1 trained
-------------------------------------------------------
Orthogonal       |     0.0336 |     0.0381 |     0.3243
Signed perm      |     0.0000 |     0.0000 |     0.0000
General inv      |     0.9909 |     0.9812 |     1.7241



## 7. Multi-seed scatter plot: regularization clusters

We train 20 autoencoders per regularization regime (different random seeds) and measure three properties of the learned encoder $W$ and decoder $W'$:

- **Orthogonality error** $\|WW^\top - I\|_F / \|I\|_F$: how far $W$'s rows are from orthonormal. Zero means $WW^\top = I$. $L_2$ regularization pushes toward orthonormal factorizations because the Frobenius penalty is minimized at the balanced, orthogonal representative of each orbit.
- **Sparsity**: fraction of entries in $W$ with $|w_{ij}| < 0.01 \cdot \max|w_{ij}|$. High sparsity means most entries are effectively zero. $L_1$ regularization drives entries to zero, selecting the sparse, axis-aligned representative.
- **Transpose error** $\|W' - W^\top\|_F / \|W^\top\|_F$: how close the decoder is to being the encoder's transpose. The transpose theorem predicts this is zero under $L_2$ regularization.

The left panel plots orthogonality error vs sparsity. The right panel plots orthogonality error vs transpose error. Each dot is one trained model. All models learn approximately the same projection $W'W$, but the specific factorization into $W$ and $W'$ differs: the regularizer selects which point on the orbit the optimizer converges to.


In [8]:
n_seeds = 20
scatter_data = {'none': [], 'l2': [], 'l1': []}

for seed in range(n_seeds):
    torch.manual_seed(seed * 13 + 7)
    for reg_type, lam in [('none', 0.0), ('l2', 0.01), ('l1', 0.005)]:
        model = LinearAutoencoder(n_dim, k_dim)
        optimizer = torch.optim.Adam(model.parameters(), lr=0.005)
        for epoch in range(2000):
            recon = model(X)
            recon_loss = ((X - recon) ** 2).mean()
            if reg_type == 'l2':
                reg = lam * (model.encoder.weight.norm()**2 + model.decoder.weight.norm()**2)
            elif reg_type == 'l1':
                reg = lam * (model.encoder.weight.abs().sum() + model.decoder.weight.abs().sum())
            else:
                reg = 0.0
            loss = recon_loss + reg
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        
        W = model.encoder.weight.detach().numpy()
        Wp = model.decoder.weight.detach().numpy()
        orth = orthogonality_measure(W)
        sp = sparsity_measure(W)
        te = transpose_error(W, Wp)
        scatter_data[reg_type].append((orth, sp, te))
    if (seed + 1) % 5 == 0:
        print(f"  Completed {seed + 1}/{n_seeds} seeds")

print("All seeds trained.")


  Completed 5/20 seeds


  Completed 10/20 seeds


  Completed 15/20 seeds


  Completed 20/20 seeds
All seeds trained.


In [9]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

colors_map = {'none': '#555555', 'l2': '#2E86C1', 'l1': '#E74C3C'}
labels_map = {'none': 'No regularization', 'l2': '$L_2$ (Frobenius)', 'l1': '$L_1$ (entry-wise)'}

# Panel 1: Orthogonality vs Sparsity
ax = axes[0]
for reg_type in ['none', 'l2', 'l1']:
    data = scatter_data[reg_type]
    orths = [d[0] for d in data]
    sps = [d[1] for d in data]
    ax.scatter(orths, sps, c=colors_map[reg_type], s=60, alpha=0.7, 
               edgecolors='white', linewidths=0.5, label=labels_map[reg_type])

ax.set_xlabel('Orthogonality error $\\|WW^\\top - I\\|_F / \\|I\\|_F$')
ax.set_ylabel('Sparsity (fraction near-zero entries)')
ax.set_title('Regularization selects different orbit representatives\n(each dot = one trained model)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

# Panel 2: Orthogonality vs Transpose error
ax = axes[1]
for reg_type in ['none', 'l2', 'l1']:
    data = scatter_data[reg_type]
    orths = [d[0] for d in data]
    tes = [d[2] for d in data]
    ax.scatter(orths, tes, c=colors_map[reg_type], s=60, alpha=0.7,
               edgecolors='white', linewidths=0.5, label=labels_map[reg_type])

ax.set_xlabel('Orthogonality error $\\|WW^\\top - I\\|_F / \\|I\\|_F$')
ax.set_ylabel("Transpose error $\\|W' - W^\\top\\| / \\|W^\\top\\|$")
ax.set_title("Transpose Theorem: $L_2$ models cluster near $W' = W^\\top$")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig('Ex3_fig_reg_multiseed_scatter.png', dpi=200, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print("Saved: Ex3_fig_reg_multiseed_scatter.png")


Saved: Ex3_fig_reg_multiseed_scatter.png


/var/folders/xh/h88wz4193gzg53_x42zm8ngm0000gn/T/ipykernel_30871/917961289.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Summary

| Regularization | Residual group | Key observation |
|---|---|---|
| None | $\mathrm{GL}_d(\mathbb{R})$ | $W'$ and $W^\top$ differ; no preferred factorization |
| $L_2$ (Frobenius) | $O_d(\mathbb{R})$ | $W' \approx W^\top$; Frobenius penalty invariant under orthogonal transforms |
| $L_1$ (entry-wise) | $B_d = \mathbb{Z}_2^d \rtimes S_d$ | Sparse weights; $L_1$ penalty invariant only under signed permutations |

The Frobenius norm's unit ball is a sphere with continuous rotational symmetry ($O(d)$). The $\ell_1$ norm's unit ball is a cross-polytope whose only symmetries are signed permutations ($B_d$). The regularizer's geometry directly determines the residual symmetry of the trained model.
